# CottonLensAI Move 3: Landsat field-level features (development years 2009–2021)

This notebook reads Landsat imagery straight from Microsoft Planetary Computer. Nothing is downloaded to Drive except small CSV files.

1. Upload the `cottonlens_move3` folder to the root of **My Drive**.
2. Choose **Runtime → Change runtime type**, then **L4 GPU** (or A100). The notebook does not use the GPU. Colab gives high RAM with these runtimes; the plain CPU runtime has much less.
3. Click **Run all**. If the connection drops, run all again: finished parts are skipped.
4. When it finishes, download `My Drive/cottonlens_move3/move3_results.zip`.

The years 2022–2025 are sealed. The script refuses them without `--unseal`.

In [ ]:
import os, sys, json, subprocess, time
!pip -q install planetary-computer pystac-client rasterio pyproj
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/cottonlens_move3'
!rm -rf /content/pkg && mkdir -p /content/pkg && unzip -q -o {BASE}/move3_package.zip -d /content/pkg
SCRIPT, PANEL = '/content/pkg/landsat_fields.py', '/content/pkg/panel.json'
ram_gb = os.sysconf('SC_PAGE_SIZE') * os.sysconf('SC_PHYS_PAGES') / 1e9
print(f'RAM {ram_gb:.0f} GB, CPU {os.cpu_count()}')

In [ ]:
import pandas as pd, numpy as np
panel = json.load(open(PANEL))['counties']

def chunks(counties, size):
    # contiguous groups inside one state, ordered west to east, so each process reads few scenes
    out = []
    for st in sorted({c['state'] for c in counties}):
        cs = sorted([c for c in counties if c['state'] == st], key=lambda c: c['centroid'][0])
        out += [cs[i:i + size] for i in range(0, len(cs), size)]
    return out

def run_chunks(groups, out_root, first, last, grid, parallel, workers):
    # one subprocess per county group; resumable because the script skips finished state-years
    todo, running, tries, failed = list(range(len(groups))), [], {}, []
    while todo or running:
        while todo and len(running) < parallel:
            k = todo.pop(0)
            d = f'{out_root}/chunk_{k:03d}'
            os.makedirs(d, exist_ok=True)
            cmd = [sys.executable, SCRIPT, PANEL, d, str(first), str(last), f'--grid={grid}', f'--workers={workers}',
                   '--fips=' + ','.join(c['fips'] for c in groups[k])]
            tries[k] = tries.get(k, 0) + 1
            running.append((k, subprocess.Popen(cmd, stdout=open(f'{d}/log.txt', 'a'), stderr=subprocess.STDOUT)))
        time.sleep(20)
        for k, p in list(running):
            if p.poll() is None:
                continue
            running.remove((k, p))
            print(f'chunk {k} try {tries[k]} exit {p.returncode}', flush=True)
            if p.returncode and tries[k] < 3:
                todo.append(k)
            elif p.returncode:
                failed.append(k)
    if failed:
        print('FAILED chunks (see their log.txt):', failed)
    return failed

## Step 1: resolution check (pre-declared, uses no yield data)
This step compares the 10 largest Texas counties at 30 m and 120 m for 2011, 2016 and 2019.
If the median correlation is ≥ 0.95, the full run uses 120 m; otherwise it uses 30 m.

In [ ]:
top = sorted([c for c in panel if c['state'] == 'TX'], key=lambda c: -c['mean_production_bales'])[:10]
RC = f'{BASE}/rescheck'
for grid, size in ((120, 10), (30, 2)):
    for y in (2011, 2016, 2019):
        run_chunks(chunks(top, size), f'{RC}/g{grid}/y{y}', y, y, grid, parallel=max(1, int(ram_gb // 12)), workers=16)

def load(root):
    fs = [os.path.join(r, f) for r, _, n in os.walk(root) for f in n if f.startswith('landsat_') and f.endswith('.csv')]
    return pd.concat([pd.read_csv(f, dtype={'fips': str}) for f in fs])

a, b = load(f'{RC}/g30'), load(f'{RC}/g120')
m = a.merge(b, on=['fips', 'year', 'cutoff'], suffixes=('_30', '_120'))
feats = ['l_p10', 'l_p25', 'l_p50', 'l_p75', 'l_p90', 'l_fail25', 'l_fail35', 'l_last', 'l_nobs']
corr = {f: float(np.corrcoef(m[f + '_30'], m[f + '_120'])[0, 1]) for f in feats}
med = float(np.median(list(corr.values())))
assert len(m) >= 60 and np.isfinite(med), f'resolution check incomplete: {len(m)} rows, median {med}; see rescheck logs'
GRID = 120 if med >= 0.95 else 30
decision = {'rows': len(m), 'corr': corr, 'median': med, 'grid_for_full_run': GRID}
json.dump(decision, open(f'{RC}/decision.json', 'w'), indent=1)
print(json.dumps(decision, indent=1))

## Step 2: full run (all 263 counties, 2009–2021)
The number of parallel processes is set from the available RAM.

In [ ]:
FULL = f'{BASE}/full_g{GRID}'
size, per_proc_gb = (12, 8) if GRID == 120 else (3, 16)
parallel = max(1, int(ram_gb // per_proc_gb))
print('groups', len(chunks(panel, size)), 'parallel', parallel)
t0 = time.time()
failed = run_chunks(chunks(panel, size), FULL, 2009, 2021, GRID, parallel=parallel, workers=16)
print('minutes', round((time.time() - t0) / 60))

In [ ]:
# completeness check and result package
n_csv = sum(1 for r, _, n in os.walk(FULL) for f in n if f.startswith('landsat_') and f.endswith('.csv'))
print('state-year CSVs:', n_csv)
!cd {BASE} && rm -f move3_results.zip && zip -q -r move3_results.zip rescheck full_g{GRID} -x '*.part' && ls -la move3_results.zip